# 05. FMEA（Failure Mode and Effects Analysis） — 練習問題

**対象技術**: 量子コンピューティング

技術がもたらす負のインパクト（失敗モード）を体系的に列挙し、深刻度 Severity・発生頻度 Occurrence・検出難易度 Detection を各1〜10で採点して、RPN = S × O × D で対策優先度をつけるリスク評価手法である。RPN は粗いスクリーニング指標であるため、クリティカリティ・マトリクスと併用して見逃しリスクを補う。

`numpy` で S/O/D 配列を扱い、`matplotlib` でクリティカリティ・マトリクスを可視化する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 故障モードの定義

量子コンピューティングの5つの故障モードについて、略称・説明・Severity・Occurrence・Detection を定義する。

In [ ]:
# (略称, 説明, Severity, Occurrence, Detection)  各 1〜10
FAILURE_MODES = [
    ("FM1", "Shorアルゴリズム実用化で RSA-2048 が危殆化",        10, 4, 3),
    ("FM2", "量子鍵配送(QKD)の実装欠陥(サイドチャネル等)",        6, 5, 5),
    ("FM3", "量子クラウドのサプライチェーン寡占依存",             7, 7, 4),
    ("FM4", "量子優位の誇大宣伝による投資バブル崩壊",             6, 6, 7),
    ("FM5", "harvest-now-decrypt-later による過去データ遡及解読", 9, 6, 9),
]

for fm in FAILURE_MODES:
    print(f"  {fm[0]}: S={fm[2]} O={fm[3]} D={fm[4]} — {fm[1]}")

## RPN の計算

S/O/D 配列から RPN = S × O × D を計算する関数を定義し、データを numpy 配列に整理する。

In [ ]:
def compute_rpn(sod):
    """S/O/D 配列(shape (n,3))から RPN = S*O*D を計算する。"""
    return sod[:, 0] * sod[:, 1] * sod[:, 2]


names = [fm[0] for fm in FAILURE_MODES]
descs = {fm[0]: fm[1] for fm in FAILURE_MODES}
sod = np.array([[fm[2], fm[3], fm[4]] for fm in FAILURE_MODES])
rpn = compute_rpn(sod)
print("RPN =", rpn.tolist())

## RPN ランキング表

故障モードを RPN の降順に並べ、対策優先順位を表示する。

In [ ]:
print("故障モード一覧と RPN:")
print(f"  {'ID':>4} | {'S':>2} | {'O':>2} | {'D':>2} | {'RPN':>4} | 説明")
print("  " + "-" * 62)
order = np.argsort(-rpn)  # RPN 降順
for rank, i in enumerate(order, start=1):
    s, o, d = sod[i]
    print(f"  {names[i]:>4} | {s:>2} | {o:>2} | {d:>2} | {rpn[i]:>4} | "
          f"{descs[names[i]]}")
print()
print("RPN 降順の対策優先順位:")
print("  " + " > ".join(f"{names[i]}({rpn[i]})" for i in order))

## クリティカリティ・マトリクス（テキスト版）

Severity × Occurrence のグリッド上に各故障モードを配置する。

In [ ]:
print("クリティカリティ・マトリクス (縦=Severity, 横=Occurrence):")
print("  S\\O " + "".join(f"{o:>4}" for o in range(1, 11)))
grid = {}
for i in range(len(names)):
    s, o = sod[i, 0], sod[i, 1]
    grid.setdefault((s, o), []).append(names[i])
for s in range(10, 0, -1):
    row = f"  {s:>3} "
    for o in range(1, 11):
        cell = grid.get((s, o))
        row += f"{cell[0]:>4}" if cell else f"{'.':>4}"
    print(row)

## まれだが激甚な事象の自動指摘

RPN は中央値以下でありながら Severity が全モード中最大の事象を「低頻度・激甚」ゾーンとして指摘する。RPN 降順だけで優先順位を決めると過小評価されるため、別枠でエスカレーションする。

In [ ]:
print("[RPN ランキングの落とし穴チェック]")
median_rpn = np.median(rpn)
max_s = sod[:, 0].max()
flagged = []
for i in range(len(names)):
    if rpn[i] <= median_rpn and sod[i, 0] == max_s:
        flagged.append(i)
if flagged:
    for i in flagged:
        print(f"  注意: {names[i]} は RPN={rpn[i]}(低順位)だが "
              f"Severity={sod[i,0]}(全モード中最大)。")
        print(f"        『低頻度・激甚』ゾーンの事象。RPN 降順だけで")
        print(f"        優先順位を決めると過小評価される。別枠でエスカレーション。")
else:
    print("  該当なし。")

## 対策後の再採点デモ（FM5）

FM5 に「PQC即時移行 + 異常検知導入」の対策を施した想定で再採点する。Severity は故障そのものの重大さであり、対策では下げられない点に注意する。

In [ ]:
print("[対策後の再採点デモ: FM5 に PQC即時移行 + 異常検知を導入]")
i5 = names.index("FM5")
before = rpn[i5]
s_a, o_a, d_a = 9, 3, 5  # 対策後: S不変, O・Dが低下
after = s_a * o_a * d_a
print(f"  対策前 RPN = {sod[i5,0]}x{sod[i5,1]}x{sod[i5,2]} = {before}")
print(f"  対策後 RPN = {s_a}x{o_a}x{d_a} = {after}  "
      f"(削減 {before - after}, {(1-after/before)*100:.0f}% 低下)")
print(f"  Severity {sod[i5,0]} は不変 — 故障そのものの重大さは対策で下げられない。")

## 可視化: クリティカリティ・マトリクス

横軸 Occurrence・縦軸 Severity の散布図に各故障モードを配置する。バブルサイズを RPN に、色を Detection（検出難易度）に対応させ、各点に故障モード ID をラベル表示する。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

occ = sod[:, 1]
sev = sod[:, 0]
det = sod[:, 2]

# バブルサイズ = RPN に比例, 色 = Detection
sizes = rpn / rpn.max() * 1800 + 200
sc = ax.scatter(occ, sev, s=sizes, c=det, cmap="YlOrRd",
                vmin=1, vmax=10, edgecolors="#333333", linewidths=1.5,
                alpha=0.85, zorder=3)

# 各点に ID と RPN をラベル
for i in range(len(names)):
    ax.annotate(f"{names[i]}\nRPN={rpn[i]}", (occ[i], sev[i]),
                ha="center", va="center", fontsize=8.5,
                fontweight="bold", zorder=4)

# 高リスク領域(右上)を薄く塗る
ax.axhspan(7.5, 10.5, xmin=0.55, color="#d1495b", alpha=0.07)
ax.text(9.6, 9.8, "high-criticality\nzone", fontsize=8,
        color="#d1495b", ha="right", va="top")

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Detection difficulty (1=easy, 10=hard)")

ax.set_title("FMEA Criticality Matrix\n"
             "(bubble size = RPN, color = Detection)", fontsize=12)
ax.set_xlabel("Occurrence")
ax.set_ylabel("Severity")
ax.set_xlim(0.5, 10.5)
ax.set_ylim(0.5, 10.5)
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

FMEA（故障モード影響解析）は、未来デザイン論文において、対象技術がもたらしうる失敗のあり方を網羅的に列挙し、それぞれを発生度・影響度・検出難易度の観点から採点して、リスク優先数（RPN）の順に並べた登録簿を作る装置として用いられる。論文は典型的に、想定される故障モードを洗い出し、各モードにスコアを与え、対策を講じるべき優先順位を提示する。結論として読者に手渡されるのは、漠然とした不安ではなく、順位づけられたリスク登録簿という構造化された警告である。

この手法がもたらす結論の型は「順位付きリスク登録簿」であり、論証はおのずと「何を防ぐか」という防御的な形をとる。境界設定の経路で見れば、結論に現れるのは故障モードとして列挙されたものに限られ、列挙されなかった失敗のあり方は分析の射程外に置かれる。価値の所在は、何を「故障」と定義し、影響度をどの利害関係者の視点で採点するかという評価の枠組みに埋め込まれる。時間観の経路では、未来は主として回避すべきリスクの集合として現れ、技術がもたらしうる便益や upside は分析の構造上ほとんど可視化されない。

同時に、この手法は固有のバイアスと方法的弱点を結論に持ち込む。失敗に焦点を絞る設計ゆえに、論文の結論は悲観的・防御的な方向に傾き、便益とリスクを衡量した総合判断には届きにくい。さらに、RPN は本来は順序尺度にすぎない三つのスコアを乗算して得られる数値であり、この方法的弱点は、発生度は低いが影響度が激甚な事象——まれだが破局的な故障——を、中程度のリスクと同等かそれ以下に評価してしまう形で結論を歪めうる。この手法を採る論文は、RPN の順位を絶対視せず、影響度が突出したモードを別枠で扱うなどの補正を施すことで、テールリスクの過小評価を避ける必要がある。

## 発展課題

**課題A**: 対策後の S/O/D（MITIGATED）を各モードに与えて再採点し、対策前後のRPN 低下量を比較せよ。S は対策で下げにくい点も確認すること。

**課題B**: RPN 乗算の限界を示す数値例を自作せよ。例えば (S,O,D)=(2,5,10) と(10,10,1) は RPN=100 で同値だが、リスクの性質は正反対である。そうした組を複数作り、RPN の単純比較がなぜ危ういか説明せよ。